# Notebook for manual implementation of Batch Norm

### Computational Graph
#### $ f_{1} = E[z_{i}] $
#### $ f_{2i} = z_{i} - f_{1} $

In [255]:
import numpy as np
import os
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import mnist1d
import random

In [256]:
def compute_f_1(z_i):
    result = np.mean(z_i)
    return result

In [257]:
def test_compute_f_1():
    input = np.array([ 2, 2, 2, 4, 4, 4 ])
    result = compute_f_1(input)
    assert(result == 3)

test_compute_f_1()

In [258]:
def compute_f_2i(z_i, f_1):
    result = z_i - f_1
    return result

In [259]:
def test_compute_f_2i():
    input = np.array([ 2, 2, 2, 4, 4, 4 ])
    result = compute_f_2i(input, compute_f_1(input))
    expected = np.array([2-3, 2-3, 2-3, 4-3, 4-3, 4-3])
    comparison = result == expected
    assert(comparison.all())

test_compute_f_2i()

### Computational Graph

#### We have the $ z_{i} $ now with the mean normalised; see `test_compute_f_2i()`

#### $ f_{3i} = f_{2i}^{2} $
#### $ f_{4} = E(f_{3i}) $
#### $ f_{5} = \sqrt{f_{4} + \epsilon} $
#### $ f_{6} = 1 / f_{5} $

In [260]:
def compute_f_3i(f_2):
    result = np.square(f_2)
    return result

def compute_f_4(f_3i):
    result = np.mean(f_3i)
    return result

def compute_f_5(f_4):
    assert(f_4 >= 0)
    epsilon = 1e-5
    result = np.sqrt(f_4 + epsilon)
    return result

def compute_f_6(f_5):
    result = 1 / f_5
    return result

In [261]:
def test_compute_f3i():
    input = np.array([2, 0, -1])
    result = compute_f_3i(input)
    expected = np.array([4, 0, 1])
    comparison = result == expected
    assert(comparison.all())

test_compute_f3i()

In [262]:
def test_compute_f_4():
    input = np.array([4, 0, 1])
    result = compute_f_4(input)
    assert((4+0+1)/3 == result)

test_compute_f_4()

In [263]:
def test_compute_f_5():
    assert(compute_f_5(0) > 0)
    result = compute_f_5(2)
    print(result)
    assert(1.41 < result)
    assert(result < 1.42)

test_compute_f_5()

1.4142170979025817


In [264]:
def test_compute_f_6():
    input = 0
    result1 = compute_f_5(input)
    assert(result1 > 0.003)
    result2 = compute_f_6(result1)
    assert(result2 > 316)
    assert(result2 < 317)

test_compute_f_6()
    

### Computational Graph

#### $ f_{7i} = f_{2i} * f_{6} $
#### $ z_{i}' = f_{7i} \times \gamma + \delta $

In [265]:
def compute_f_7i(f_2i, f_6):
    result = f_2i * f_6
    return result

def compute_z_i_prime(f_7i, gamma, delta):
    result = f_7i * gamma + delta
    return result

In [266]:
def test_compute_f_7i():
    result = compute_f_7i(2, 4)
    assert(result == 8)

def test_compute_z_i_prime():
    result = compute_z_i_prime(-0.2, 5, 3)
    assert(result == 2)

test_compute_f_7i()
test_compute_z_i_prime()

In [267]:
def test_overall():
    z_i = np.array([ 2, 2, 2, 4, 4, 4 ])
    gamma = 5
    delta = 2
    f_1 = compute_f_1(z_i)
    f_2i = compute_f_2i(z_i, f_1)
    f_3i = compute_f_3i(f_2i)
    f_4 = compute_f_4(f_3i)
    f_5 = compute_f_5(f_4)
    f_6 = compute_f_6(f_5)
    f_7i = compute_f_7i(f_2i, f_6)
    z_i_prime = compute_z_i_prime(f_7i, gamma, delta)
    print(z_i_prime)
    assert(z_i_prime[0] + 3 < 0.1)
    assert(z_i_prime[5] - 7 >-0.1)

test_overall()
    


[-2.999975 -2.999975 -2.999975  6.999975  6.999975  6.999975]


### Cross check
https://kratzert.github.io/2016/02/12/understanding-the-gradient-flow-through-the-batch-normalization-layer.html

In [268]:
def batchnorm_forward(x, gamma, beta, eps):

  N, D = x.shape

  #step1: calculate mean
  mu = 1./N * np.sum(x, axis = 0)

  #step2: subtract mean vector of every trainings example
  xmu = x - mu

  #step3: following the lower branch - calculation denominator
  sq = xmu ** 2

  #step4: calculate variance
  var = 1./N * np.sum(sq, axis = 0)

  #step5: add eps for numerical stability, then sqrt
  sqrtvar = np.sqrt(var + eps)

  #step6: invert sqrtwar
  ivar = 1./sqrtvar

  #step7: execute normalization
  xhat = xmu * ivar

  #step8: Nor the two transformation steps
  gammax = gamma * xhat

  #step9
  out = gammax + beta

  #store intermediate
  cache = (xhat,gamma,xmu,ivar,sqrtvar,var,eps)

  return out, cache

In [269]:
def test_batchnorm_forward():
    x = np.array([ 2, 2, 2, 4, 4, 4 ])
    gamma = 5
    delta = 2
    beta = delta
    eps = 1e-5

    # Reshape to 6 rows of 1 dimensional column
    x = x.reshape(-1, 1)
    result = batchnorm_forward(x, gamma, beta, eps)
    print (result[0])

test_batchnorm_forward()

[[-2.999975]
 [-2.999975]
 [-2.999975]
 [ 6.999975]
 [ 6.999975]
 [ 6.999975]]


In [270]:
def batchnorm_backward(dout, cache):

  #unfold the variables stored in cache
  xhat,gamma,xmu,ivar,sqrtvar,var,eps = cache

  #get the dimensions of the input/output
  N,D = dout.shape

  #step9
  dbeta = np.sum(dout, axis=0)
  dgammax = dout #not necessary, but more understandable

  #step8
  dgamma = np.sum(dgammax*xhat, axis=0)
  dxhat = dgammax * gamma

  #step7
  divar = np.sum(dxhat*xmu, axis=0)
  dxmu1 = dxhat * ivar

  #step6
  dsqrtvar = -1. /(sqrtvar**2) * divar

  #step5
  dvar = 0.5 * 1. /np.sqrt(var+eps) * dsqrtvar

  #step4
  dsq = 1. /N * np.ones((N,D)) * dvar

  #step3
  dxmu2 = 2 * xmu * dsq

  #step2
  dx1 = (dxmu1 + dxmu2)
  dmu = -1 * np.sum(dxmu1+dxmu2, axis=0)

  #step1
  dx2 = 1. /N * np.ones((N,D)) * dmu

  #step0
  dx = dx1 + dx2

  return dx, dgamma, dbeta

In [271]:
def test_batch_norm_backward():
    x = np.array([ 2, 2, 2, 4, 4, 4 ])
    gamma = 5
    delta = 2
    beta = delta
    eps = 1e-5

    # Reshape to 6 rows of 1 dimensional column
    x = x.reshape(-1, 1)
    result = batchnorm_forward(x, gamma, beta, eps)
    result2 = batchnorm_backward(result[0], result[1])
    print(result2)
    
test_batch_norm_backward()

(array([[-0.00025],
       [-0.00025],
       [-0.00025],
       [ 0.00025],
       [ 0.00025],
       [ 0.00025]]), array([29.9997]), array([12.]))
